# Chapter 2: Turn domain work into executable contracts

Companion to *Agent Harness Engineering*. Listings 2.1 to 2.7 live in `ahe/ch02.py`.

## Setup

In [ ]:
import sys, pathlib
root = pathlib.Path.cwd()
if not (root / "ahe").exists():
    root = root.parent            # opened from inside chapter-0N/
sys.path.insert(0, str(root))
assert sys.version_info >= (3, 10), "Use Python 3.10 or newer."
import os
LIVE = os.environ.get("AHE_LIVE") == "1"
print("Python", sys.version.split()[0], "| live calls:", "on" if LIVE else "off")

## 1. The fixtures (Table 2.5)

In [ ]:
from ahe import ch02
from ahe.ch02 import FIXTURES, run_fixtures, prepare, request, ground, NOTES
failures = run_fixtures(FIXTURES)
assert failures == {}, failures
print(len(FIXTURES), "fixtures pass.")

## 2. A packet, with the quotation the reviewer sees

In [ ]:
from pprint import pprint
pprint(prepare(request("P036", 3)))

## 3. The hostile readers against `ground`

In [ ]:
from datetime import date
on = date(2026, 9, 1)
note = NOTES["P036"][0]
for reader in (ch02.fake_reader, ch02.liar, ch02.forger, ch02.drifter, ch02.redater,
               ch02.flipper, ch02.fragment, ch02.hider, ch02.garbler):
    proposal = reader({"note": note.text, "field": "anticoagulant"})
    print(f"{reader.__name__:12s} -> {ground(proposal, note, 'anticoagulant', on) or 'grounded'}")

## 4. The recorded live reader (Table 2.2)

In [ ]:
from collections import Counter
from ahe import ch01, live
by_text = {n.text: (pid, n) for pid, ns in NOTES.items() for n in ns}
recs = live.records("ch02_reader_gpt-6-luna")
tally = Counter()
for r in recs:
    pid, n = by_text[r["prompt"].split("Note: ", 1)[1]]
    try:
        p = ch01.parse_json(r["reply"])
    except ValueError:
        p = {}
    tally[(pid, p.get("status"), ground(p, n, "anticoagulant", date(2026, 9, 1)) or "grounded")] += 1
for k, v in sorted(tally.items()):
    print(k, v)
print(len(recs), "calls; cost $%.4f" % sum(r["cost_usd"] for r in recs))

### Live calls

Every live cell below does nothing unless `AHE_LIVE=1` is set before Jupyter starts. Settings come from the environment or from `~/.config/ahe/openrouter.env`:

```
AHE_BASE_URL=https://openrouter.ai/api/v1     # or https://<workspace>/serving-endpoints
AHE_API_KEY=...                                # never commit this file
AHE_MODEL=openai/gpt-6-luna
```

Any OpenAI-compatible endpoint works, including a Databricks workspace (base URL `https://<workspace-host>/serving-endpoints`, a personal access token as the key, and a served model name). `ahe.live.Recorder` replaces `chat` so that each call is appended to `runs/<experiment>.jsonl` with its prompt, reply, token counts, and cost. Use a new experiment name for your own runs so the book's records stay as they are.

In [ ]:
if LIVE:
    live.load_env()
    ch01.chat = live.Recorder("my_ch02_reader")
    for pid in ("P036", "P037", "P038", "P039"):
        got = prepare(request(pid, 3), model=ch02.live_reader)
        print(pid, got["criteria"]["no_anticoag"])
else:
    print("Set AHE_LIVE=1 to run the live reader.")

## Exercise solutions

### Exercise 2.1

Version 2 has no freshness limit, so the June 2025 reading is `met`; version 3 limits readings to 90 days, so the same reading is `unknown`, stale. The protocol contract changed, not the evidence. The packet names the version, and the reason names the reading's date, so two packets a month apart cannot be confused.

In [ ]:
for v in (2, 3):
    print(v, prepare(request("P023", v))["criteria"]["marker"])

### Exercise 2.2

The coordinator would not pick one: two fresh readings that disagree on the status are `unknown`, and the reason names both. Write the fixture first, then change `evaluate`.

In [ ]:
from datetime import date
from ahe.ch02 import Evidence, Result, evaluate, finite, PROTOCOLS

def evaluate_2_2(rule, evidence, on):
    result = evaluate(rule, evidence, on)
    if result.status == "unknown":
        return result          # missing, stale, not a number, or wrong unit
    fresh = [e for e in evidence if e.field == rule.field and e.unit == rule.unit
             and finite(e.value) and e.observed <= on and (rule.max_age_days is None
                                       or (on - e.observed).days <= rule.max_age_days)]
    statuses = {rule.lower <= e.value <= rule.upper for e in fresh}
    if len(statuses) > 1:
        return Result("unknown", "fresh readings disagree: "
                      + ", ".join(f"{e.value} on {e.observed}" for e in fresh))
    return result

marker = PROTOCOLS[1].rules[1]
readings = [Evidence("x1", "marker", 5, "ng/mL", date(2026, 8, 20)),
            Evidence("x2", "marker", 9, "ng/mL", date(2026, 8, 25))]
print(evaluate(marker, readings, date(2026, 9, 1)))
print(evaluate_2_2(marker, readings, date(2026, 9, 1)))
broken = [Evidence("x3", "marker", None, "ng/mL", date(2026, 8, 20)), readings[0]]
print(evaluate_2_2(marker, broken, date(2026, 9, 1)))   # unknown: not a number

### Exercise 2.3

The stand-in reader sees `warfarin` and a date and reports `present`; the quotation and date are both in the note, so `ground` accepts it and the rule comes out `not met`. The packet's quotation is what lets the reviewer catch the error. Measuring how often it happens needs notes labeled by a person, which Chapter 9 builds.

In [ ]:
from ahe.ch02 import Note, read_note
tricky = Note("n99", date(2026, 8, 1),
              "Warfarin was considered on 2026-08-01 and rejected because of bleeding risk.")
print(read_note(ch02.fake_reader, tricky, "anticoagulant", date(2026, 9, 1))[:2])